In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- noaa_ghcn_inventory ---
FIX_NOAA_GHCN_INVENTORY_INVENTORY_FILE = "test_file.csv"

# --- noaa_ghcn_stations ---
FIX_NOAA_GHCN_STATIONS_LISTINGS_FILE = "test_file.csv"

NOAA_TEST_FILE_TEXT = """ACW00011604  17.1167  -61.7833   10.1 ?  ST JOHNS COOLIDGE FLD                  1949 2013 4
US1ALNB0006  34.4842  -86.4533  241.7 2  HORTON 0.1 ENE                                     
"""

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_noaa_ghcn_inventory(inventory_file):
    inventory_df = pd.read_fwf(inventory_file, header=None, colspecs="infer", infer_nrows=np.inf)
    inventory_df = pl.from_pandas(inventory_df)
    inventory_df = inventory_df[:, [0, 4, 5]]
    return inventory_df

def before_noaa_ghcn_stations(listings_file):
    df = pd.read_fwf(
        listings_file,
        dtype=str,
        header=None,
        colspecs="infer",
        infer_nrows=np.inf,
    )
    df = pl.from_pandas(df)
    df = df[:, [0, 1, 2, 3, 4, 5, 8]]
    return df

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_noaa_ghcn_inventory(inventory_file):
    import numpy as np

    inventory_df = pd.read_fwf(inventory_file, header=None, colspecs="infer", infer_nrows=np.inf)
    inventory_df = pl.from_pandas(inventory_df)
    inventory_df = inventory_df[:, [0, 4, 5]]
    return inventory_df

def gen_noaa_ghcn_stations(listings_file):
    import numpy as np

    df = pd.read_fwf(
        listings_file,
        dtype=str,
        header=None,
        colspecs="infer",
        infer_nrows=np.inf,
    )
    df = pl.from_pandas(df)
    df = df[:, [0, 1, 2, 3, 4, 5, 8]]
    return df

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:

# === Tests: noaa_ghcn_stations ===
import tempfile

def _make_noaa_file(text=NOAA_TEST_FILE_TEXT):
    tmp = tempfile.TemporaryDirectory()
    path = Path(tmp.name) / "noaa.txt"
    path.write_text(text, encoding="utf-8")
    return tmp, path

try:
    _tmp, _path = _make_noaa_file()
    try: _r = gen_noaa_ghcn_stations(str(_path))
    finally: _tmp.cleanup()
    print("✅ L1 smoke gen_noaa_ghcn_stations: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_noaa_ghcn_stations: {type(_e).__name__}: {_e}")

try:
    _tmp, _path = _make_noaa_file()
    try: _rb = before_noaa_ghcn_stations(str(_path))
    finally: _tmp.cleanup()
    print("✅ L1 smoke before_noaa_ghcn_stations: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_noaa_ghcn_stations: {type(_e).__name__}: {_e}")

try:
    _tmp, _path = _make_noaa_file()
    try:
        _rb = before_noaa_ghcn_stations(str(_path)); _rg = gen_noaa_ghcn_stations(str(_path))
    finally: _tmp.cleanup()
    compare(_rb, _rg, "noaa_ghcn_stations", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence noaa_ghcn_stations: setup error — {type(_e).__name__}: {_e}")

try:
    _tmp, _path = _make_noaa_file(NOAA_TEST_FILE_TEXT.splitlines()[0] + "\n")
    try:
        _rb = before_noaa_ghcn_stations(str(_path)); _rg = gen_noaa_ghcn_stations(str(_path))
    finally: _tmp.cleanup()
    compare(_rb, _rg, "L3 edge noaa_ghcn_stations single row", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge noaa_ghcn_stations single row: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_noaa_ghcn_stations: OK, type= DataFrame
✅ L1 smoke before_noaa_ghcn_stations: OK
✅ L2 equivalence noaa_ghcn_stations: MATCH
✅ L3 edge noaa_ghcn_stations single row: MATCH
